In [1]:
# =======================================================================
# FILE: 1_mega_scraper_clean.ipynb
# DESCRIPTION: Google News RSS Scraper Interaktif (Modular, Clean & Robust)
# =======================================================================

import sys
import subprocess
from IPython.display import clear_output, display

def install_dependencies():
    packages = ["feedparser", "pandas", "requests", "ipywidgets", "urllib3"]
    subprocess.check_call([sys.executable, "-m", "pip", "install", *packages, "--quiet"])
    clear_output()

print("🔄 Menyiapkan sistem dan memeriksa library...")
install_dependencies()

import re
import time
import urllib.parse
from datetime import datetime, timedelta, date
from typing import List, Tuple, Dict

import feedparser
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import ipywidgets as widgets


# =======================================================================
# KONFIGURASI & DAFTAR KATA KUNCI
# =======================================================================
DEFAULT_QUERIES: List[str] = [
    "huru hara", "kerusuhan", "provokasi", "hoaks", "bentrok", "tawuran",
    "disinformasi", "fake news", "media sosial ricuh", "fitnah",
    "pencemaran nama baik", "ujaran kebencian", "cyber patrol", "patroli siber",
    "peretasan", "hacker", "fitnah siber", "massa terprovokasi", "unjuk rasa",
    "demonstrasi ricuh", "klarifikasi hoaks", "turnbackhoax", "kominfo cek fakta",
    "penipuan online", "phishing", "scam", "judi online", "judol",
    "akun palsu", "penculikan", "begal", "pembunuhan", "provokator", "black campaign"
]


# =======================================================================
# MODUL PEMROSESAN TEKS & TANGGAL
# =======================================================================
class TextCleaner:
    @staticmethod
    def strip_source_from_title(raw_title: str) -> str:
        return re.sub(r'\s+-\s+[^-]+$', '', raw_title).strip()

    @staticmethod
    def clean_for_nlp(text: str) -> str:
        text = str(text).lower()
        text = re.sub(r'http\S+|www\S+|https\S+', '', text)
        text = re.sub(r'[^\w\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text


def generate_date_chunks(start_date: date, end_date: date, delta_days: int) -> List[Tuple[date, date]]:
    if delta_days <= 0:
        return [(start_date, end_date)]

    chunks = []
    curr_date = start_date
    while curr_date <= end_date:
        next_date = min(curr_date + timedelta(days=delta_days - 1), end_date)
        chunks.append((curr_date, next_date))
        curr_date = next_date + timedelta(days=1)
    return chunks


# =======================================================================
# ENGINE SCRAPER UTAMA
# =======================================================================
class GoogleNewsScraper:
    def __init__(self, delay: float = 0.4):
        self.delay = delay
        self.base_url = "https://news.google.com" + "/rss/search"
        self.session = self._create_session()

    def _create_session(self) -> requests.Session:
        session = requests.Session()
        session.headers.update({
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                          "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        })
        retries = Retry(total=3, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
        session.mount("https://", HTTPAdapter(max_retries=retries))
        return session

    def fetch_rss(self, query: str, start_dt: date, end_dt: date) -> List[Dict]:
        inclusive_end = end_dt + timedelta(days=1)
        search_q = f'"{query}" after:{start_dt} before:{inclusive_end}'
        params = {
            "q": search_q,
            "hl": "id",
            "gl": "ID",
            "ceid": "ID:id"
        }
        url = f"{self.base_url}?{urllib.parse.urlencode(params)}"

        response = self.session.get(url, timeout=15)
        response.raise_for_status()
        feed = feedparser.parse(response.content)

        results = []
        for entry in feed.entries:
            raw_title = entry.get("title", "")
            clean_title = TextCleaner.strip_source_from_title(raw_title)
            source = entry.source.title if hasattr(entry, "source") else "Unknown Source"

            results.append({
                "keyword": query,
                "text": clean_title,
                "raw_title": raw_title,
                "source": source,
                "date": entry.get("published", ""),
                "url": entry.get("link", "")
            })
        return results

    def run_scraping(self, queries: List[str], date_chunks: List[Tuple[date, date]]) -> pd.DataFrame:
        scraped_data = []
        seen_titles = set()

        for c_idx, (s_chunk, e_chunk) in enumerate(date_chunks, 1):
            print(f"\n🔥 [PERIODE {c_idx}/{len(date_chunks)}] Rentang: {s_chunk} s/d {e_chunk}")

            for q_text in queries:
                try:
                    entries = self.fetch_rss(q_text, s_chunk, e_chunk)
                    new_count = 0

                    for item in entries:
                        if item["text"] not in seen_titles:
                            seen_titles.add(item["text"])
                            scraped_data.append(item)
                            new_count += 1

                    if new_count > 0:
                        print(f"   ✓ '{q_text}' -> +{new_count} berita baru")

                except Exception as e:
                    print(f"   ❌ Gagal pada '{q_text}': {e}")

                time.sleep(self.delay)

        return pd.DataFrame(scraped_data)


# =======================================================================
# ANTARMUKA PENGGUNA (UI WIDGETS) & EKSEKUSI
# =======================================================================
class ScraperDashboard:
    def __init__(self, queries: List[str]):
        self.queries = queries
        self.scraper = GoogleNewsScraper(delay=0.4)

        self.start_picker = widgets.DatePicker(
            description='Tgl Mulai:',
            value=datetime.now().date() - timedelta(days=90)
        )
        self.end_picker = widgets.DatePicker(
            description='Tgl Selesai:',
            value=datetime.now().date()
        )
        self.chunk_dropdown = widgets.Dropdown(
            options=[
                ('Tanpa Chunking (Cepat)', 0),
                ('Per 3 Bulan (90 Hari)', 90),
                ('Per 1 Bulan (30 Hari - Maksimal Data)', 30),
                ('Per 2 Minggu (14 Hari - Super Detail)', 14),
                ('Per 1 Minggu (7 Hari - Ultra Detail)', 7)
            ],
            value=14,
            description='Mode Tarik:'
        )
        self.run_btn = widgets.Button(
            description='Mulai Tarik Data',
            button_style='success',
            icon='download'
        )
        self.output_area = widgets.Output()
        self.run_btn.on_click(self._on_run_clicked)

    def _preprocess_and_save(self, df: pd.DataFrame, start_dt: date, end_dt: date):
        print("\n🧹 [TAHAP AKHIR] Membersihkan teks & menghapus duplikat...")
        df['clean_text'] = df['text'].apply(TextCleaner.clean_for_nlp)
        df = df.drop_duplicates(subset=['clean_text'])
        df = df[df['clean_text'].str.len() > 5].reset_index(drop=True)

        df['parsed_date'] = pd.to_datetime(df['date'], errors='coerce').dt.strftime('%Y-%m-%d %H:%M:%S')

        file_latest = 'dataset_cyber_alerts_LATEST.csv'
        file_dated = f'dataset_cyber_{start_dt}_to_{end_dt}_FINAL.csv'

        df.to_csv(file_latest, index=False, encoding='utf-8-sig')
        df.to_csv(file_dated, index=False, encoding='utf-8-sig')

        print("\n" + "=" * 65)
        print("🎉 PROSES SELESAI & DATA BERHASIL DISIMPAN!")
        print("=" * 65)
        print(f"📁 File Sistem   : {file_latest}")
        print(f"📁 File Arsip    : {file_dated}")
        print(f"📊 Total Bersih  : {len(df)} artikel unik")
        print("=" * 65)

    def _on_run_clicked(self, _):
        with self.output_area:
            clear_output()
            start_dt = self.start_picker.value
            end_dt = self.end_picker.value
            chunk_days = self.chunk_dropdown.value

            if not start_dt or not end_dt:
                print("❌ Error: Harap pilih Tanggal Mulai dan Tanggal Selesai.")
                return
            if start_dt > end_dt:
                print("❌ Error: Tanggal Mulai tidak boleh lebih besar dari Tanggal Selesai.")
                return

            date_chunks = generate_date_chunks(start_dt, end_dt, chunk_days)
            print(f"🚀 MEMULAI SCRAPING ({len(self.queries)} kata kunci | {len(date_chunks)} sub-periode)")
            print(f"📅 Rentang Utama: {start_dt} s/d {end_dt}")
            print("=" * 65)

            df_result = self.scraper.run_scraping(self.queries, date_chunks)

            if df_result.empty:
                print("\n⚠️ Tidak ditemukan berita pada rentang tanggal tersebut.")
                return

            self._preprocess_and_save(df_result, start_dt, end_dt)

    def render(self):
        print("✅ Sistem siap! Atur rentang tanggal dan klik tombol di bawah:")
        ui_box = widgets.VBox([
            widgets.HBox([self.start_picker, self.end_picker]),
            self.chunk_dropdown,
            self.run_btn,
            self.output_area
        ])
        display(ui_box)


# =======================================================================
# EKSEKUSI DASHBOARD
# =======================================================================
dashboard = ScraperDashboard(queries=DEFAULT_QUERIES)
dashboard.render()


✅ Sistem siap! Atur rentang tanggal dan klik tombol di bawah:
